### HANDLING IMBALANCED DATASET

**What if your dataset is imbalanced?**

- Imagine 95% of your rows are Class 0 and only 5% are Class 1. 
- A model could achieve 95% accuracy by simply predicting Class 0 every single time — without learning anything useful. 
- This is the **class imbalance problem**, and this notebook covers **four** techniques to handle it.

In [2]:
#Importing libraries
import pandas as pd
import numpy as np

#Loading the data
data = pd.read_csv("Synergix_data_preprocessed_new.csv")

#Storing the ratio in a list named Rating_ratio
Rating_ratio = []
for row in data.values:
    if(row[4]+row[5] == 0):
        if(row[7]+row[8] == 0):
            #If all the ratings are zero then overall rating ratio will also be zero
            Rating_ratio.append(0.0)
        else:
            #If only the numerator(1 and 2 star) ratings are zero then adding -99999 to the list temporarily which
            #will be taken care of in the next cell.
            Rating_ratio.append(-99999)
    else:
        Rating_ratio.append((int(row[7])+(row[8]))/(int(row[4])+int(row[5])))

#replacing -99999 with the maximum ratio in the list
max_rating = max(Rating_ratio)
for x in range(len(Rating_ratio)):
    if(Rating_ratio[x] == -99999):
        Rating_ratio[x] = max_rating

#adding the column 'Good_By_Bad_Rating' to the dataframe
data['Good_By_Bad_Rating'] = Rating_ratio

data = data.drop(columns = ['1_Star_Rating', '2_Star_Rating', '3_Star_Rating', '4_Star_Rating', '5_Star_Rating'])

In [3]:
from sklearn.preprocessing import LabelEncoder
data [['Segment']]= data [['Segment']].apply(LabelEncoder().fit_transform)

In [4]:
data = data.drop(columns = 'Units_sold',axis=1)

X = data.drop(columns = 'Units_sold>1000')
y = data['Units_sold>1000']

In [5]:
# Importing the train-test split from scikit-learn
from sklearn.model_selection import train_test_split

# Performing train and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30, random_state = 6)

#### Checking the Class Ratio Before Stratification


In [6]:
y_train.value_counts(normalize=True)

Units_sold>1000
1    0.598137
0    0.401863
Name: proportion, dtype: float64

In [7]:
y_test.value_counts(normalize=True)

Units_sold>1000
1    0.614261
0    0.385739
Name: proportion, dtype: float64

#### Stratified Train-Test Split

In [8]:
# Performing train test split with stratification
X_train_st, X_test_st, y_train_st, y_test_st = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=6
)

### *stratify=y*

**What stratification does**: 
- Instead of splitting rows randomly, it splits within each class separately. 
- If your original data is 60% Class 0 and 39% Class 1, then both your train and test sets will also be exactly 60/39. 
- The proportions are guaranteed to be identical across splits.

In [9]:
y_train_st.value_counts(normalize=True)

Units_sold>1000
1    0.60294
0    0.39706
Name: proportion, dtype: float64

In [10]:
y_test_st.value_counts(normalize=True)

Units_sold>1000
1    0.603056
0    0.396944
Name: proportion, dtype: float64

**This is an important lesson**: 
- On large datasets, random splitting usually gets you close to equal proportions anyway — the law of large numbers takes care of it. 
- But on small datasets, random splits can produce very skewed distributions. 
- **Stratification is cheap and always the safer habit**, so you should use it by default for any classification problem.

#### Class Weights

In [11]:
from sklearn.tree import DecisionTreeClassifier
DT_model = DecisionTreeClassifier(
    max_depth=11, min_samples_leaf=6,
    random_state=42,
    class_weight='balanced'
)

However, we can also assign customised class weights by defining a dictionary where the keys represent the class labels, and #the values represent the weights we want to assign to each class.

DT_model = DecisionTreeClassifier(max_depth = 11, min_samples_leaf= 6, random_state=42, class_weight = {0:0.8, 1:0.2})

In [12]:
# Train the model
DT_model.fit(X_train_st, y_train_st)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",11
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",6
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

#### Predictions and Evaluation

In [13]:
from sklearn.metrics import f1_score

# Make predictions on the train dataset
y_train_pred = DT_model.predict(X_train_st)

# Make predictions on the test dataset
y_test_pred = DT_model.predict(X_test_st)

# Let's display the model performance on the train and test data.

print('Train score: ', f1_score(y_train_st, y_train_pred))
print('Test score: ', f1_score(y_test_st, y_test_pred))

Train score:  0.8870789957134109
Test score:  0.8261986301369864


---

### Undersampling

Now that we have implemented the tricks to handle class imbalance, let's dive deeper to understand how to balance the dataset using undersampling and oversampling techniques. First, let's do undersampling!

To do so, we have to first install a ski-kit learn library called 'imblearn'. It is specifically designed to deal with imbalanced datasets and help us seamlessly implement various methods like undersampling, oversampling, and SMOTE etc. Write the following code to install the same!

In [1]:
#pip install --upgrade scikit-learn imbalanced-learn
!pip install imblearn


   ---------------------------------------- 0/3 [sklearn-compat]
   ---------------------------------------- 0/3 [sklearn-compat]
   ---------------------------------------- 0/3 [sklearn-compat]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ongja\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Once the library is installed, let's import RandomUnderSampler from imblearn.under_sampling

In [14]:
from imblearn.under_sampling import RandomUnderSampler

#### Applying Random Undersampling

- *.fit_resample(X_train, y_train)*: this is the key method. It looks at y_train, counts how many samples exist per class, then randomly removes rows from the majority class until both classes are equal. It returns the reduced *X_train_rus* and *y_train_rus*.

**Critical detail — only applied to training data.** You never undersample your test set. The test set must remain untouched and representative of real-world data — otherwise your evaluation is misleading.

In [15]:
sampler = RandomUnderSampler(random_state = 42)
X_train_rus, y_train_rus = sampler.fit_resample(X_train, y_train)

#### Verifying the Balance

In [16]:
print(y_train_rus.value_counts(normalize = True))

Units_sold>1000
0    0.5
1    0.5
Name: proportion, dtype: float64


As you can see from the ratios, the class distribution is equal now due to random undersampling.
Let's build the model using these modified training data and see its performance!

In [17]:
DT_model = DecisionTreeClassifier(max_depth = 11, min_samples_leaf= 6, random_state=42)

DT_model.fit(X_train_rus, y_train_rus)

y_train_pred = DT_model.predict(X_train_rus)
y_pred = DT_model.predict(X_test)


print('Train F1 Score: ', f1_score(y_train_rus, y_train_pred))
print('Test F1 Score: ', f1_score(y_test, y_pred))

Train F1 Score:  0.8667085539897675
Test F1 Score:  0.8229858504187121


**The tradeoff of undersampling**: 
- You gain class balance but lose data volume. 
- On **large** datasets the loss is tolerable. 
- On **small** datasets, undersampling can be very damaging.

#### Random Oversampling

Performing random oversampling involves doing the same steps like we did for random undersampler. However, the only difference is we have to import RandomOverSampler from imblearn.over_sampling and build the model.

Let's perform resampling of the orginial data.

In [19]:
from imblearn.over_sampling import RandomOverSampler

sampler = RandomOverSampler(random_state = 42)
X_train_ros, y_train_ros = sampler.fit_resample(X_train, y_train)

The *.fit_resample()* method now works in the opposite direction: randomly duplicating minority class rows.

In [20]:
y_train_ros.value_counts(normalize = True)

Units_sold>1000
1    0.5
0    0.5
Name: proportion, dtype: float64

As you can see, it's equal. Let's go to the next stepn and build the model now.

In [21]:
DT_model = DecisionTreeClassifier(max_depth = 11, min_samples_leaf= 6, random_state=42)

DT_model.fit(X_train_ros, y_train_ros)

y_train_pred = DT_model.predict(X_train_ros)
y_pred = DT_model.predict(X_test)

y_train_pred = DT_model.predict(X_train_ros)
y_pred = DT_model.predict(X_test)

# Printing the F1 score of the train and test data
print('Train F1 Score: ', f1_score(y_train_ros, y_train_pred))
print('Test F1 Score: ', f1_score(y_test, y_pred))

Train F1 Score:  0.8634057971014493
Test F1 Score:  0.8295983086680762


### Observation:
- Slightly better than undersampling — because no information was discarded. 
- The majority class data is fully intact. 
- The **downside** is that duplicated rows can cause the model to overfit on those specific minority examples.

#### SMOTE (The Smart Oversampling)

**How SMOTE works internally**:
```python
Existing minority point A: [Segment=1, Page_traffic=2000, ...]
Existing minority point B: [Segment=1, Page_traffic=3000, ...]
SMOTE creates:             [Segment=1, Page_traffic=2400, ...]  ← interpolated, not real
```

In [22]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42)

X_train_smt, y_train_smt = smote.fit_resample(X_train,y_train)

Let's quickly check the ratio of the classes in the training data.

In [23]:
y_train_smt.value_counts(normalize = True)

Units_sold>1000
1    0.5
0    0.5
Name: proportion, dtype: float64

Let's build the decision tree model using the oversampled training data.

In [ ]:
DT_model = DecisionTreeClassifier(max_depth = 11, min_samples_leaf= 6, random_state=42)
DT_model.fit(X_train_smt, y_train_smt)


#Making predictions
y_train_pred = DT_model.predict(X_train_smt)
y_pred = DT_model.predict(X_test)

#Evaluating the model
print('Training F1 score: ', f1_score(y_train_smt, y_train_pred))
print('Testing F1 score: ', f1_score(y_test, y_pred))

| Technique | What it changes | Risk |
|-|-|-|
| Stratified split | How data is divided | None — just better practice |
| Class weights | How splits are scored internally | None — data unchanged |
| Undersampling | Removes majority rows | Information loss |
| Random Oversampling | Duplicates minority rows | Overfitting on repeated samples |
| SMOTE | Generates synthetic minority rows	 | Synthetic data may not be realistic |